In [ ]:
# Notebook 17i: RF Features — Synth Only (cGAN) (path_mnist)
import os, numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
SEED=42; np.random.seed(SEED)
CLASS_NAMES=['Adipose', 'Background', 'Debris', 'Lymphocytes', 'Mucus', 'Smooth Muscle', 'Normal Colon Mucosa', 'Cancer-Assoc. Stroma', 'Colorectal Adenocarcinoma']
CRITERION='gini'; MAX_DEPTH=100; N_ESTIMATORS=100
PROJECT_DIR='/content/drive/MyDrive/thesis_project'
CGAN_FEAT=os.path.join(PROJECT_DIR,'data','extracted_features','seed_42','cgan')
REAL_FEAT=os.path.join(PROJECT_DIR,'data','extracted_features','seed_42','real')
SYNTH_NPZ=os.path.join(PROJECT_DIR,'data','synthetic','seed_42','cgan','path_mnist/path_mnist_synthetic.npz')
REAL_NPZ=os.path.join(PROJECT_DIR,'data','processed','seed_42','path_mnist.npz')
RESULTS_DIR=os.path.join(PROJECT_DIR,'results','rf_features','path_mnist','synth_only_cgan')
os.makedirs(RESULTS_DIR,exist_ok=True)

In [ ]:
F_tr=np.concatenate([
    np.load(os.path.join(CGAN_FEAT,'path_mnist_x_train_features.npy')),
    np.load(os.path.join(CGAN_FEAT,'path_mnist_x_val_features.npy'))],axis=0)
synth=np.load(SYNTH_NPZ)
y_tr=np.concatenate([synth['y_train'],synth['y_val']],axis=0)
F_test=np.load(os.path.join(REAL_FEAT,'path_mnist_x_test_features.npy'))
y_test=np.load(REAL_NPZ)['y_test']
print(f'F_tr: {F_tr.shape}, F_test: {F_test.shape}')

In [ ]:
clf=RandomForestClassifier(criterion=CRITERION,max_depth=MAX_DEPTH,
    n_estimators=N_ESTIMATORS,random_state=SEED,n_jobs=-1)
clf.fit(F_tr,y_tr)
y_pred=clf.predict(F_test)
test_acc=accuracy_score(y_test,y_pred)
print(f'Test accuracy: {test_acc:.4f}')
print(classification_report(y_test,y_pred,target_names=CLASS_NAMES,digits=4))
report_dict = classification_report(y_test, y_pred,
    target_names=CLASS_NAMES, digits=4, output_dict=True)
macro_f1    = round(report_dict['macro avg']['f1-score'], 4)
weighted_f1 = round(report_dict['weighted avg']['f1-score'], 4)
pd.DataFrame(report_dict).transpose().to_csv(
    os.path.join(RESULTS_DIR, 'classification_report.csv'))
pd.DataFrame([{
    'condition':   'synth_only_cgan',
    'dataset':     'path_mnist',
    'test_acc':    round(test_acc, 4),
    'macro_f1':    macro_f1,
    'weighted_f1': weighted_f1,
}]).to_csv(os.path.join(RESULTS_DIR, 'summary.csv'), index=False)

In [ ]:
# ── Final Cell: Save Predictions & Confusion Matrix ──────────────────────────
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

y_prob = clf.predict_proba(F_test)
y_pred = clf.predict(F_test)

np.save(os.path.join(RESULTS_DIR, 'y_pred.npy'), y_pred)
np.save(os.path.join(RESULTS_DIR, 'y_test.npy'), y_test)
np.save(os.path.join(RESULTS_DIR, 'y_prob.npy'), y_prob)

cm = confusion_matrix(y_test, y_pred)
pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
    os.path.join(RESULTS_DIR, 'confusion_matrix.csv'))
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix — path_mnist | synth_only_cgan')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.close()
print('All outputs saved.')